# StreamMind — GPU Evaluation on Google Colab

This notebook runs the StreamMind evaluation pipeline on a Colab GPU runtime and produces LaTeX-ready numbers for the paper tables.

**What it does:**
1. Installs dependencies, clones the repo, and sets deterministic seeds
2. Verifies GPU availability (A100 recommended for paper-quality latency)
3. **Phase 1** — Latency profiling (per-component breakdown on GPU)
4. **Phase 2** — LiveQA-Bench evaluation (55 streams, 2500+ QA pairs, 3 scopes) — *primary results*
5. **Phase 3** — Ablation study (memory sizes, FIFO, no-TQR)
6. **Phase 4** — Published reference numbers (OVO-Bench, EgoSchema from cited papers)
7. Converts all results to LaTeX table values

**Note:** OVO-Bench and EgoSchema baseline numbers come from published papers (Li et al. CVPR 2025; He et al. CVPR 2025). Only LiveQA-Bench and ablations require GPU evaluation.

**Runtime:** Select **A100 GPU** under *Runtime → Change runtime type* for best results. T4 also works but produces higher latency numbers.

**Reproducibility:** All random seeds are fixed (seed=42). The canonical benchmark (`liveqa_bench.json`) is committed to the repo and reused across runs so that results are deterministic.

| Phase | Estimated Time (T4) | Estimated Time (A100) |
|-------|--------------------|-----------------------|
| Latency profiling | ~2 min | ~30 sec |
| LiveQA-Bench (full) | ~5 min | ~1 min |
| Full ablation (6 configs) | ~30 min | ~8 min |

## 0. Setup

In [ ]:
# @title Clone the repository and install dependencies
import os

# ---------- CONFIGURE ----------
REPO_URL = "https://github.com/palsure/StreamMind-LiveQA.git"
BRANCH = "main"
# --------------------------------

PROJECT_ROOT = "/content/StreamMind-LiveQA"

if os.path.exists(PROJECT_ROOT):
    !cd {PROJECT_ROOT} && git pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {PROJECT_ROOT}

# Install Python dependencies
!pip install -q -r {PROJECT_ROOT}/eval/requirements.txt rouge-score
!pip install -q -r {PROJECT_ROOT}/demo/backend/requirements.txt

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Contents: {os.listdir(PROJECT_ROOT)}")

In [ ]:
# @title Alternative: Upload code as zip
# Uncomment this cell if you prefer uploading a zip instead of cloning.

# from google.colab import files
# uploaded = files.upload()  # Upload StreamMind-LiveQA.zip
# !unzip -q StreamMind-LiveQA.zip -d /content/
# PROJECT_ROOT = "/content/StreamMind-LiveQA"

In [ ]:
# @title Verify GPU, set seed, and set up paths
import torch
import sys
import random
import numpy as np

assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → GPU (A100 recommended)."
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

if "A100" not in gpu_name:
    print(f"\nWARNING: A100 GPU recommended for paper-quality latency numbers.")
    print(f"  Current GPU ({gpu_name}) will work but produce higher latencies.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"\nRandom seed: {SEED} (deterministic mode)")

# Add backend to path so eval scripts can import StreamMind modules
BACKEND_DIR = f"{PROJECT_ROOT}/demo/backend"
EVAL_DIR = f"{PROJECT_ROOT}/eval"
RESULTS_DIR = f"{PROJECT_ROOT}/eval/results"

for p in [BACKEND_DIR, EVAL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["STREAMMIND_PROJECT_ROOT"] = PROJECT_ROOT

print(f"\nBackend: {BACKEND_DIR}")
print(f"Results: {RESULTS_DIR}")

In [ ]:
# @title Pre-download models (prevents timeout during evaluation)
print("Downloading models (first run only, ~2 GB)...")

from transformers import (
    CLIPModel, CLIPProcessor,
    BlipForConditionalGeneration, BlipProcessor,
    BlipForQuestionAnswering,
    T5ForConditionalGeneration, T5Tokenizer,
)

models = [
    ("CLIP ViT-B/32", lambda: (
        CLIPModel.from_pretrained("openai/clip-vit-base-patch32"),
        CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32"),
    )),
    ("BLIP Captioning", lambda: (
        BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base"),
        BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base"),
    )),
    ("BLIP VQA", lambda: (
        BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base"),
        BlipProcessor.from_pretrained("Salesforce/blip-vqa-base"),
    )),
    ("Flan-T5-base", lambda: (
        T5ForConditionalGeneration.from_pretrained("google/flan-t5-base"),
        T5Tokenizer.from_pretrained("google/flan-t5-base"),
    )),
]

for name, loader in models:
    print(f"  {name}...", end=" ", flush=True)
    loader()
    print("done")

print("\nAll models cached.")

## 1. Latency Profiling (GPU)

Measures per-component latency on GPU. These numbers replace the CPU estimates in the paper's latency table.

In [ ]:
# @title Download sample videos (demo + evaluation clips)
import subprocess, shutil
from pathlib import Path

# Run the bundled download script with --eval flag to get all clips
subprocess.run(
    [sys.executable, f"{PROJECT_ROOT}/demo/scripts/download_samples.py", "--eval"],
    check=True,
)

# Verify
samples_dir = Path(PROJECT_ROOT) / "demo" / "frontend" / "samples"
video_files = sorted(samples_dir.glob("*.mp4"))
print(f"\nFound {len(video_files)} sample video(s):")
total_mb = 0
for v in video_files:
    size_mb = v.stat().st_size / (1024 * 1024)
    total_mb += size_mb
    print(f'  {v.name:30s} {size_mb:6.1f} MB')
print(f'  {"TOTAL":30s} {total_mb:6.1f} MB')

In [ ]:
# @title Run latency profiling
import importlib, run_docker_eval as rde
importlib.reload(rde)
from run_docker_eval import profile_latency, set_seed, _resolve_paths

set_seed(SEED)

# Re-resolve paths for Colab
rde.RESULTS_DIR, rde.VIDEOS = _resolve_paths(PROJECT_ROOT)
print(f"Videos available: {list(rde.VIDEOS.keys())}")
print(f"Results dir: {rde.RESULTS_DIR}")

latency = profile_latency(n_runs=20)

print("\n" + "=" * 55)
print(f"  GPU Latency Results ({gpu_name})")
print("=" * 55)
for comp, v in latency.items():
    print(f"  {comp:22s}  {v['mean_ms']:8.1f} ms  (std {v['std_ms']:.1f})")
print("=" * 55)

In [ ]:
# @title Format latency for paper table
import json

with open(f"{RESULTS_DIR}/latency.json") as f:
    lat = json.load(f)

print("% Paste into paper/sec/4_experiments.tex — Tab:latency")
print("% GPU column (measured on", gpu_name, ")")

mapping = [
    ("CLIP encoding (per frame)", "clip_encode"),
    ("SKM scoring + update", "skm_update"),
    ("TQR scope classification", "tqr_classify"),
    ("BLIP captioning (per frame)", "blip_caption"),
    ("BLIP VQA (per frame)", "blip_vqa"),
    ("Flan-T5 synthesis", "flan_t5"),
]

print()
for label, key in mapping:
    if key in lat:
        ms = lat[key]["mean_ms"]
        if ms < 1:
            val = "$<$1"
        else:
            val = f"{ms:.0f}" if ms >= 10 else f"$\\sim${ms:.0f}"
        print(f"    {label:35s} & {val} \\\\")
    else:
        print(f"    {label:35s} & TBD \\\\")

## 2. LiveQA-Bench Evaluation

Builds the LiveQA-Bench from sample videos and evaluates StreamMind (full model).

In [ ]:
# @title Load canonical LiveQA-Bench and evaluate full model
import importlib, sys

# Force-reload ALL backend modules so code changes take effect
for mod_name in ['memory_manager', 'stream_processor', 'vlm_engine', 'run_docker_eval']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from run_docker_eval import build_liveqa_bench, evaluate_liveqa, set_seed
from vlm_engine import VLMEngine
import run_docker_eval as rde
rde.RESULTS_DIR, rde.VIDEOS = rde._resolve_paths(PROJECT_ROOT)

# Verify the fixes are loaded
from memory_manager import MemoryManager
import inspect
src = inspect.getsource(MemoryManager.get_entries_by_scope)
has_fallback = 'min_recent_frames' in src
print(f'memory_manager fallback fix loaded: {has_fallback}')
from vlm_engine import MAX_SAMPLE_FRAMES
print(f'vlm_engine MAX_SAMPLE_FRAMES: {MAX_SAMPLE_FRAMES}')

set_seed(SEED)
vlm = VLMEngine()

# Set force_rebuild=True on first run to generate the benchmark from all
# available videos.  On subsequent runs, set to False to reuse the saved
# benchmark for deterministic, comparable results across GPUs.
FORCE_REBUILD = True  # Set to False after the first successful run

qa_list = build_liveqa_bench(vlm, force_rebuild=FORCE_REBUILD)
del vlm

print(f"\nLoaded {len(qa_list)} QA pairs")
from collections import Counter
scope_counts = Counter(q.scope for q in qa_list)
print(f"Scope distribution: {dict(scope_counts)}")

# Run full model evaluation
full_results = evaluate_liveqa(qa_list, memory_capacity=64, label="full")

In [ ]:
# @title Display LiveQA-Bench results
print("=" * 55)
print("  LiveQA-Bench — Full Model (N=64)")
print("=" * 55)
print(f"  Overall accuracy: {full_results['overall_accuracy']:.1f}%")
print(f"  Average score:    {full_results['avg_score']:.3f}")
print(f"  Average latency:  {full_results['avg_latency_ms']:.0f} ms")
print()
print("  Per-scope accuracy:")
for scope, acc in full_results["per_scope_accuracy"].items():
    n = full_results["scope_counts"][scope]
    print(f"    {scope:15s}  {acc:5.1f}%  (n={n})")
print("=" * 55)

# LaTeX line for paper
sa = full_results["per_scope_accuracy"]
oa = full_results["overall_accuracy"]
print("\n% LiveQA table row (Tab:liveqa):")
print(f"    \\textbf{{\\ours}} & \\textbf{{{sa.get('instant', 'TBD')}}} "
      f"& \\textbf{{{sa.get('recent', 'TBD')}}} "
      f"& \\textbf{{{sa.get('historical', 'TBD')}}} "
      f"& \\textbf{{{oa}}} \\\\")

In [ ]:
# @title Debug: inspect recent-scope predictions
import json
from pathlib import Path

lf = Path(rde.RESULTS_DIR) / 'liveqa_full.json'
if lf.exists():
    with open(lf) as f:
        data = json.load(f)
    recent = [r for r in data['results'] if r['scope'] == 'recent']
    print(f'{len(recent)} recent-scope results:\n')
    for r in recent[:6]:
        print(f"  Q: {r['question']}")
        print(f"  GT: {r['ground_truth'][:80]}")
        print(f"  Pred: {r['predicted'][:80]}")
        print(f"  Score: {r['score']}  Correct: {r['correct']}")
        print()

## 3. Ablation Study

Tests different configurations on LiveQA-Bench: memory sizes (N=16, 32, 128), FIFO buffer, no TQR.

In [ ]:
# @title Run ablation study
from run_docker_eval import run_ablations, set_seed

set_seed(SEED)
ablation_results = run_ablations(qa_list)

In [ ]:
# @title Display ablation results + LaTeX
print("=" * 60)
print("  Ablation Study Results")
print("=" * 60)

for name, res in ablation_results.items():
    print(f"  {name:12s}: acc={res['overall_accuracy']:5.1f}%  "
          f"score={res['avg_score']:.3f}  "
          f"scope_acc={res['per_scope_accuracy']}")

print("\n% Ablation table (Tab:ablation):")

label_map = {
    "full":   r"\textbf{\ours (full model)}",
    "fifo":   r"\quad w/o SKM (FIFO buffer, $N\!=\!64$)",
    "no_tqr": r"\quad w/o TQR (attend to full memory always)",
    "N16":    r"\quad $N = 16$ memory slots",
    "N32":    r"\quad $N = 32$ memory slots",
    "N128":   r"\quad $N = 128$ memory slots",
}

for key in ["full", "fifo", "no_tqr", "N16", "N32", "N128"]:
    if key in ablation_results:
        acc = ablation_results[key]["overall_accuracy"]
        label = label_map.get(key, key)
        bold = r"\textbf{" + f"{acc}" + "}" if key == "full" else str(acc)
        print(f"    {label:55s} & {bold} \\\\")

## 3b. OVO-Bench Evaluation (Optional)

Evaluates StreamMind directly on OVO-Bench (CVPR 2025). Requires OVO-Bench videos (~44 GB).

**Skip this section** if you only need LiveQA-Bench results. Published baseline numbers are already in the paper.

### Option A: Download directly in Colab (needs ~50 GB free disk)
Run the download cell below. May fail if Colab disk is full.

### Option B: Download locally → Google Drive → Colab (recommended)
1. On your local machine, download `src_videos.tar.parta[a-e]` from [HuggingFace](https://huggingface.co/datasets/JoeLeelyf/OVO-Bench)
2. Concatenate and extract: `cat src_videos.tar.part* | tar xf -`
3. Upload the extracted video files to Google Drive (e.g. `My Drive/ovobench/videos/`)
4. Run the **Google Drive mount** cell below — it will set `OVOBENCH_DIR` to your Drive path

In [ ]:
# @title Option A: Download OVO-Bench annotations and videos directly in Colab
# Videos are ~44 GB total. Skip this cell if using Google Drive (Option B).
from pathlib import Path

OVOBENCH_DIR = f"{PROJECT_ROOT}/eval/benchmarks/data/ovobench"
os.makedirs(f"{OVOBENCH_DIR}/videos", exist_ok=True)

# Download annotations from HuggingFace
print("Downloading OVO-Bench annotations...")
try:
    from datasets import load_dataset
    ds = load_dataset("JoeLeelyf/OVO-Bench", split="test")
    print(f"  Loaded {len(ds)} questions from HuggingFace")

    items = [dict(row) for row in ds]
    with open(f"{OVOBENCH_DIR}/annotations.json", "w") as f:
        json.dump(items, f, indent=2)
    print(f"  Saved annotations.json ({len(items)} entries)")
except Exception as e:
    print(f"  Annotation download failed: {e}")
    print("  Download manually: https://huggingface.co/datasets/JoeLeelyf/OVO-Bench")

# Check for videos
n_vids = len(list(Path(OVOBENCH_DIR).rglob("*.mp4")))
print(f"\n  Videos available: {n_vids}")
if n_vids == 0:
    print("  No videos found. Either:")
    print("    1. Download in Colab (needs ~50 GB free disk):")
    print("       !pip install -q huggingface_hub")
    print("       !huggingface-cli download JoeLeelyf/OVO-Bench --local-dir ovobench_dl")
    print(f"    2. Or use Option B: upload to Google Drive and run the next cell")

In [ ]:
# @title Option B: Mount Google Drive for OVO-Bench videos (recommended)
# Use this if you downloaded OVO-Bench videos locally and uploaded them to Google Drive.
#
# Steps:
#   1. Download src_videos.tar.parta[a-e] from HuggingFace on your local machine
#   2. Extract: cat src_videos.tar.part* | tar xf -
#   3. Upload the extracted videos/ folder to Google Drive, e.g. My Drive/ovobench/videos/
#   4. Run this cell — it mounts Drive and points OVOBENCH_DIR to your videos

from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# --- CONFIGURE THIS PATH ---
# Change this to match where you put the videos in Google Drive
GDRIVE_OVOBENCH = "/content/drive/MyDrive/ovobench"
# ----------------------------

OVOBENCH_DIR = GDRIVE_OVOBENCH
n_vids = len(list(Path(OVOBENCH_DIR).rglob("*.mp4"))) if Path(OVOBENCH_DIR).exists() else 0
print(f"\nOVO-Bench videos found in Google Drive: {n_vids}")

if n_vids == 0:
    print(f"\nNo videos found at: {GDRIVE_OVOBENCH}")
    print("Make sure you uploaded the extracted videos to this Google Drive path.")
    print("Expected structure:")
    print(f"  {GDRIVE_OVOBENCH}/videos/*.mp4")
    print(f"  {GDRIVE_OVOBENCH}/annotations.json  (or download via Option A)")
else:
    print(f"Ready to evaluate! Run the next cell.")

# Download annotations if not present in Drive
anno_path = Path(OVOBENCH_DIR) / "annotations.json"
if not anno_path.exists():
    print("\nAnnotations not found in Drive, downloading from HuggingFace...")
    try:
        from datasets import load_dataset
        ds = load_dataset("JoeLeelyf/OVO-Bench", split="test")
        items = [dict(row) for row in ds]
        anno_path.parent.mkdir(parents=True, exist_ok=True)
        with open(anno_path, "w") as f:
            json.dump(items, f, indent=2)
        print(f"  Saved annotations.json ({len(items)} entries)")
    except Exception as e:
        print(f"  Failed: {e}")

In [ ]:
# @title Run StreamMind on OVO-Bench
# Only runs if OVO-Bench videos are available.
from pathlib import Path

n_vids = len(list(Path(OVOBENCH_DIR).rglob("*.mp4")))
if n_vids > 0:
    import importlib
    for mod_name in ['evaluate', 'pipeline', 'metrics', 'benchmarks.ovobench']:
        if mod_name in sys.modules:
            importlib.reload(sys.modules[mod_name])

    from evaluate import evaluate_benchmark
    from pipeline import EvalPipeline

    pipeline = EvalPipeline(
        memory_capacity=64,
        sample_fps=2.0,
        frame_skip=1,
        device="cuda",
    )

    print(f"Evaluating StreamMind on OVO-Bench ({n_vids} videos)...")
    try:
        ovo_results = evaluate_benchmark(
            "ovobench", OVOBENCH_DIR, pipeline,
            max_samples=0,
            compute_gpt_score=False,
            output_dir=RESULTS_DIR,
        )
        print(f"\n  OVO-Bench accuracy: {ovo_results['accuracy']}%")
        if 'per_category' in ovo_results:
            for cat, acc in ovo_results['per_category'].items():
                print(f"    {cat}: {acc}%")

        # LaTeX row for OVO-Bench table
        pc = ovo_results.get('per_category', {})
        bt = pc.get('backward_tracing', pc.get('BT', '?'))
        rt = pc.get('real_time', pc.get('RT', '?'))
        fa = pc.get('forward_active', pc.get('FA', '?'))
        oa = ovo_results['accuracy']
        print(f"\n% OVO-Bench table row:")
        print(f"    \\textbf{{\\ours}} & {bt} & {rt} & {fa} & {oa} \\\\")
    except Exception as e:
        print(f"  OVO-Bench evaluation failed: {e}")
        import traceback; traceback.print_exc()
else:
    print("No OVO-Bench videos found. Skipping evaluation.")
    print("See the cell above for download instructions.")

In [ ]:
# @title Summary: all results for the paper
import json
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 70)
print(f"  StreamMind Evaluation Summary — {gpu_name}")
print("=" * 70)

# LiveQA-Bench
print(f"\n--- LiveQA-Bench (N=64) ---")
sa = full_results["per_scope_accuracy"]
print(f"  Overall: {full_results['overall_accuracy']:.1f}%")
print(f"  Instant: {sa.get('instant', 'N/A')}%  |  Recent: {sa.get('recent', 'N/A')}%  |  Historical: {sa.get('historical', 'N/A')}%")
print(f"  Avg latency: {full_results['avg_latency_ms']:.0f} ms")

# Ablation
print(f"\n--- Ablation ---")
for key in ["full", "fifo", "no_tqr", "N16", "N32", "N128"]:
    if key in ablation_results:
        r = ablation_results[key]
        s = r['per_scope_accuracy']
        print(f"  {key:8s}: overall={r['overall_accuracy']:5.1f}%  "
              f"I={s.get('instant','?'):>5}  R={s.get('recent','?'):>5}  H={s.get('historical','?'):>5}  "
              f"lat={r['avg_latency_ms']:.0f}ms")

# Latency
lat_path = results_path / "latency.json"
if lat_path.exists():
    with open(lat_path) as f:
        lat = json.load(f)
    print(f"\n--- Latency Breakdown ({gpu_name}) ---")
    for comp, v in lat.items():
        print(f"  {comp:22s}: {v['mean_ms']:7.1f} ms  (std {v['std_ms']:.1f})")

print(f"\n{'=' * 70}")
print(f"  Results saved to: {RESULTS_DIR}")
print(f"  Download the zip in Section 6 to update the paper.")
print(f"{'=' * 70}")

## 4. Published Reference Numbers (OVO-Bench & EgoSchema)

Instead of running external benchmarks (which require large video downloads), we use **published results** from peer-reviewed papers. These numbers go directly into the paper tables.

| Benchmark | Source Paper | Numbers Used |
|-----------|-------------|-------------|
| OVO-Bench | Li et al., CVPR 2025 | Table 1 (all streaming + offline baselines) |
| EgoSchema | He et al. (Dispider), CVPR 2025 | Table 2 (full-video baselines) |

The cell below prints the published numbers in LaTeX-ready format.

In [ ]:
# @title 4a. Published reference numbers for paper tables
# These numbers come from peer-reviewed publications and go directly into
# the paper's OVO-Bench table and the EgoSchema reference paragraph.

print("=" * 65)
print("  OVO-Bench — Published Results (Li et al., CVPR 2025)")
print("  Source: Table 1 of arXiv:2501.05510v2")
print("=" * 65)

ovo_published = {
    "Offline (full video)": {
        "Qwen2-VL-7B":          {"BT": 46.5, "RT": 56.0, "FA": 48.7, "Overall": 50.4},
        "LLaVA-OneVision-7B":   {"BT": 43.7, "RT": 64.0, "FA": 50.5, "Overall": 52.7},
    },
    "Streaming (causal)": {
        "Flash-VStream":        {"BT": 27.4, "RT": 28.4, "FA": 45.1, "Overall": 33.6},
        "VideoLLM-online":      {"BT": 17.7, "RT": 20.8, "FA": "---", "Overall": "---"},
        "Dispider":             {"BT": 36.1, "RT": 54.6, "FA": 34.7, "Overall": 41.8},
    },
}

for group, models in ovo_published.items():
    print(f"\n  {group}:")
    for name, vals in models.items():
        print(f"    {name:25s}  BT={vals['BT']}  RT={vals['RT']}  FA={vals['FA']}  Overall={vals['Overall']}")

print("\n" + "=" * 65)
print("  EgoSchema — Published Full-Video Results (He et al., CVPR 2025)")
print("  Source: Table 2 of arXiv:2501.03218 (Dispider paper)")
print("=" * 65)

ego_published = {
    "Video-LLaVA":       38.4,
    "LLaVA-Next-Video":  43.9,
    "Dispider":          55.6,
}

for name, acc in ego_published.items():
    print(f"  {name:25s}  Acc = {acc}%")

print("\n" + "-" * 65)
print("  These numbers are already in paper/sec/4_experiments.tex.")
print("  No GPU evaluation needed for external benchmarks.")
print("-" * 65)

In [ ]:
# @title [SKIP] 4b. (Legacy) Download NExT-QA — no longer needed
# External benchmark numbers now come from published papers.
# This cell is kept for reference only; do not run it.
pass

BENCH_ROOT = f"{PROJECT_ROOT}/eval/benchmarks/data"
NEXTQA_DIR = f"{BENCH_ROOT}/nextqa"
os.makedirs(f"{NEXTQA_DIR}/videos", exist_ok=True)

print("=" * 60)
print("  NExT-QA — Downloading annotations")
print("=" * 60)

# Download annotation CSVs directly from the NExT-QA GitHub repo
import urllib.request

NEXTQA_GH = "https://raw.githubusercontent.com/doc-doc/NExT-QA/main/dataset/nextqa"
for split_file in ["val.csv", "test.csv", "train.csv"]:
    dest = f"{NEXTQA_DIR}/{split_file}"
    if not os.path.exists(dest):
        url = f"{NEXTQA_GH}/{split_file}"
        print(f"  Downloading {split_file}...", end=" ", flush=True)
        try:
            urllib.request.urlretrieve(url, dest)
            print("done")
        except Exception as e:
            print(f"failed: {e}")
    else:
        print(f"  {split_file} already exists")

# Count questions in val split
val_csv = f"{NEXTQA_DIR}/val.csv"
if os.path.exists(val_csv):
    with open(val_csv) as f:
        n_val = sum(1 for _ in csv.reader(f)) - 1
    print(f"\n  val.csv: {n_val} questions")

# Collect unique video IDs from val split
video_ids = set()
if os.path.exists(val_csv):
    with open(val_csv, newline="") as f:
        for row in csv.DictReader(f):
            video_ids.add(row["video"])
    print(f"  Unique videos needed: {len(video_ids)}")

print("\n" + "=" * 60)
print("  NExT-QA — Downloading videos")
print("=" * 60)
print("  NExT-QA videos come from VidOR (YFCC100M).")
print("  Attempting download via yt-dlp for available clips...")
print("  Note: Many old YFCC100M videos are no longer available.")

# Try the NExT-QA video download approach:
# The videos are typically named by their VidOR ID (e.g., "1234")
# and are short clips (avg 44s). We try yt-dlp as a fallback.
# If videos can't be downloaded, we can run annotation-only validation.

# Check how many videos we already have
existing = set(
    p.stem for p in Path(f"{NEXTQA_DIR}/videos").glob("*.mp4")
)
missing = video_ids - existing
print(f"  Already have: {len(existing)} videos")
print(f"  Missing: {len(missing)} videos")

if missing:
    print(f"\n  To get NExT-QA videos, use one of these methods:")
    print(f"  1. Download VidOR dataset: https://xdshang.github.io/docs/vidor.html")
    print(f"  2. Clone NExT-QA repo and run their download script:")
    print(f"     git clone https://github.com/doc-doc/NExT-QA.git")
    print(f"     python NExT-QA/videos/download.py")
    print(f"  3. Upload videos manually to: {NEXTQA_DIR}/videos/")
    print(f"\n  Skipping NExT-QA video eval for now (annotations ready).")

In [ ]:
# @title [SKIP] 4c. (Legacy) Download EgoSchema — no longer needed
# EgoSchema numbers now come from the Dispider paper (He et al., CVPR 2025).
pass

EGOSCHEMA_DIR = f"{BENCH_ROOT}/egoschema"
os.makedirs(f"{EGOSCHEMA_DIR}/videos", exist_ok=True)

print("=" * 60)
print("  EgoSchema — Downloading annotations from HuggingFace")
print("=" * 60)

from datasets import load_dataset

try:
    ds = load_dataset("lmms-lab/egoschema", split="test")
    print(f"  Loaded {len(ds)} questions from HuggingFace")
    print(f"  Columns: {ds.column_names}")

    # Convert to the JSON format expected by our benchmark loader
    questions = {}
    subset_answers = {}
    for row in ds:
        q_uid = row.get("q_uid", row.get("id", ""))
        entry = {
            "q_uid": q_uid,
            "question": row.get("question", ""),
        }
        for i in range(5):
            key = f"option {i}"
            if key in row:
                entry[key] = row[key]
            elif f"option_{i}" in row:
                entry[f"option {i}"] = row[f"option_{i}"]
        if "answer" in row:
            entry["answer"] = row["answer"]
            subset_answers[q_uid] = row["answer"]
        if "video_uid" in row:
            entry["video_uid"] = row["video_uid"]

        questions[q_uid] = entry

    with open(f"{EGOSCHEMA_DIR}/questions.json", "w") as f:
        json.dump(questions, f, indent=2)
    print(f"  Saved questions.json ({len(questions)} entries)")

    with open(f"{EGOSCHEMA_DIR}/subset_answers.json", "w") as f:
        json.dump(subset_answers, f, indent=2)
    print(f"  Saved subset_answers.json ({len(subset_answers)} entries)")

except Exception as e:
    print(f"  HuggingFace download failed: {e}")
    print("  Trying direct download from GitHub...")
    import urllib.request
    for fname in ["questions.json", "subset_answers.json"]:
        url = f"https://raw.githubusercontent.com/egoschema/EgoSchema/main/{fname}"
        dest = f"{EGOSCHEMA_DIR}/{fname}"
        try:
            urllib.request.urlretrieve(url, dest)
            print(f"  Downloaded {fname}")
        except Exception as e2:
            print(f"  Failed to download {fname}: {e2}")

# --- Video download via Kaggle ---
print("\n" + "=" * 60)
print("  EgoSchema — Downloading videos")
print("=" * 60)

existing_vids = list(Path(f"{EGOSCHEMA_DIR}/videos").glob("*.mp4"))
print(f"  Existing videos: {len(existing_vids)}")

if len(existing_vids) < 100:
    print("\n  Option 1: Kaggle (recommended)")
    print("  ─────────────────────────────────")
    print("  1. Go to https://www.kaggle.com/settings → Create API Token")
    print("  2. Upload kaggle.json when prompted below")
    print("  3. Accept competition rules at: https://www.kaggle.com/competitions/egoschema/rules")

    try:
        # Check if kaggle.json already exists
        kaggle_dir = os.path.expanduser("~/.kaggle")
        kaggle_json = f"{kaggle_dir}/kaggle.json"

        if not os.path.exists(kaggle_json):
            print("\n  Upload your kaggle.json file:")
            from google.colab import files
            uploaded = files.upload()
            os.makedirs(kaggle_dir, exist_ok=True)
            for fname, content in uploaded.items():
                with open(kaggle_json, "wb") as f:
                    f.write(content)
            os.chmod(kaggle_json, 0o600)
            print("  kaggle.json saved!")

        print("  Downloading EgoSchema videos from Kaggle...")
        subprocess.run(
            ["pip", "install", "-q", "kaggle"],
            check=True, capture_output=True,
        )
        result = subprocess.run(
            ["kaggle", "competitions", "download", "-c", "egoschema",
             "-p", f"{EGOSCHEMA_DIR}/videos"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            # Unzip downloaded files
            for zf in Path(f"{EGOSCHEMA_DIR}/videos").glob("*.zip"):
                subprocess.run(["unzip", "-q", "-o", str(zf), "-d",
                                f"{EGOSCHEMA_DIR}/videos"], check=True)
                zf.unlink()
            n_vids = len(list(Path(f"{EGOSCHEMA_DIR}/videos").glob("*.mp4")))
            print(f"  Downloaded {n_vids} videos!")
        else:
            print(f"  Kaggle download failed: {result.stderr[:200]}")
            print("  Make sure you accepted the competition rules.")

    except KeyboardInterrupt:
        print("\n  Skipped Kaggle upload.")
    except Exception as e:
        print(f"  Kaggle download error: {e}")

    print("\n  Option 2: Manual upload")
    print("  ───────────────────────")
    print(f"  Upload .mp4 files to: {EGOSCHEMA_DIR}/videos/")
else:
    print(f"  {len(existing_vids)} videos found — ready for evaluation!")

In [ ]:
# @title [SKIP] 4d. (Legacy) Download OVO-Bench — no longer needed
# OVO-Bench numbers now come from Li et al. (CVPR 2025).
pass

OVOBENCH_DIR = f"{BENCH_ROOT}/ovobench" if 'BENCH_ROOT' in dir() else ""
os.makedirs(f"{OVOBENCH_DIR}/videos", exist_ok=True)

print("=" * 60)
print("  OVO-Bench — Downloading annotations")
print("=" * 60)

try:
    ds = load_dataset("JoeLeelyf/OVO-Bench", split="test")
    print(f"  Loaded {len(ds)} questions from HuggingFace")
    print(f"  Columns: {ds.column_names}")

    # Save as JSON for our benchmark loader
    items = []
    for row in ds:
        items.append(dict(row))
    with open(f"{OVOBENCH_DIR}/val_questions.json", "w") as f:
        json.dump(items, f, indent=2)
    print(f"  Saved val_questions.json ({len(items)} entries)")

except Exception as e:
    print(f"  Failed: {e}")
    print("  Try: git clone https://github.com/JoeLeelyf/OVO-Bench.git")

print(f"\n  OVO-Bench videos are ~44 GB (source) or ~144 GB (chunked).")
print(f"  To download, use the OVO-Bench repo scripts:")
print(f"    git clone https://github.com/JoeLeelyf/OVO-Bench.git")
print(f"    # Then download src_videos.tar.parta[a-e] from HuggingFace:")
print(f"    # https://huggingface.co/datasets/JoeLeelyf/OVO-Bench")
print(f"  Place videos in: {OVOBENCH_DIR}/videos/")

In [ ]:
# @title [SKIP] 4e. (Legacy) Verify downloads — no longer needed
pass
from pathlib import Path

DATA_CONFIG = {
    "nextqa":    NEXTQA_DIR,
    "egoschema": EGOSCHEMA_DIR,
    "ovobench":  OVOBENCH_DIR,
}

print("=" * 60)
print("  Benchmark Data Status")
print("=" * 60)

ready_benchmarks = []
for name, root in DATA_CONFIG.items():
    root_p = Path(root)
    vid_dir = root_p / "videos"
    n_videos = len(list(vid_dir.glob("*.mp4"))) if vid_dir.exists() else 0

    # Check annotations
    if name == "nextqa":
        anno = root_p / "val.csv"
    elif name == "egoschema":
        anno = root_p / "questions.json"
    elif name == "ovobench":
        anno = root_p / "val_questions.json"
    else:
        anno = None

    has_anno = anno.exists() if anno else False
    status = "READY" if (has_anno and n_videos > 0) else (
        "ANNOTATIONS ONLY" if has_anno else "NOT AVAILABLE"
    )

    symbol = {
        "READY": "\u2705", "ANNOTATIONS ONLY": "\u26a0\ufe0f", "NOT AVAILABLE": "\u274c"
    }.get(status, "?")

    print(f"  {symbol} {name:15s}  {status:20s}  ({n_videos} videos)")
    if status == "READY":
        ready_benchmarks.append(name)

print(f"\nReady for evaluation: {ready_benchmarks if ready_benchmarks else 'None'}")
if not ready_benchmarks:
    print("Upload videos to the benchmark directories above to run evaluation.")

In [ ]:
# @title [SKIP] 4f. (Legacy) Run external benchmark evaluation — no longer needed
pass
# Set MAX_SAMPLES to a small number (e.g., 50) for a quick test run.

MAX_SAMPLES = 0  # 0 = evaluate all samples; set to 50 for quick test

import importlib, sys
for mod_name in ['evaluate', 'pipeline', 'metrics']:
    if mod_name in sys.modules:
        importlib.reload(sys.modules[mod_name])

from evaluate import evaluate_benchmark
from pipeline import EvalPipeline

pipeline = EvalPipeline(
    memory_capacity=64,
    sample_fps=2.0,
    frame_skip=1,
    device="cuda",
)

ext_results = {}

for bench_name in ready_benchmarks:
    data_root = DATA_CONFIG[bench_name]
    print(f"\n{'=' * 60}")
    print(f"  Evaluating: {bench_name}")
    print(f"{'=' * 60}")

    kwargs = {}
    if bench_name == "egoschema":
        kwargs["subset"] = "subset"

    try:
        summary = evaluate_benchmark(
            bench_name, data_root, pipeline,
            max_samples=MAX_SAMPLES,
            compute_gpt_score=False,
            output_dir=RESULTS_DIR,
            **kwargs,
        )
        ext_results[bench_name] = summary
        print(f"\n  {bench_name} accuracy: {summary['accuracy']}%")
    except Exception as e:
        print(f"  Error on {bench_name}: {e}")
        import traceback; traceback.print_exc()

# Save combined results
if ext_results:
    with open(f"{RESULTS_DIR}/external_summaries.json", "w") as f:
        json.dump(ext_results, f, indent=2)
    print(f"\nSaved to {RESULTS_DIR}/external_summaries.json")

    print("\n" + "=" * 60)
    print("  External Benchmark Results — LaTeX")
    print("=" * 60)
    for bname, s in ext_results.items():
        acc = s.get("accuracy", "?")
        print(f"  {bname}: {acc}%")
        if "per_type" in s:
            for t, v in s["per_type"].items():
                print(f"    {t}: {v}%")
        if "per_category" in s:
            for c, v in s["per_category"].items():
                print(f"    {c}: {v}%")
        if "gpt_score" in s:
            print(f"    GPT Score: {s['gpt_score']}")
        if "recall_at_1_iou03" in s:
            print(f"    R@1 (IoU>=0.3): {s['recall_at_1_iou03']}%")
else:
    print("\nNo external benchmarks evaluated (no data ready).")
    print("See cells 4b-4d above to download benchmark data.")

## 5. Generate LaTeX Values for the Paper

Reads all result JSON files and prints the exact values to replace `\tbd` placeholders in `paper/sec/4_experiments.tex`.

In [ ]:
# @title Generate all LaTeX values
import json
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 60)
print("  LaTeX Values for paper/sec/4_experiments.tex")
print("=" * 60)

# --- LiveQA-Bench (from run_docker_eval output) ---
liveqa_path = results_path / "liveqa_full.json"
if liveqa_path.exists():
    with open(liveqa_path) as f:
        liveqa = json.load(f)["summary"]
    sa = liveqa["per_scope_accuracy" if "per_scope_accuracy" in liveqa else "per_scope"]
    oa = liveqa["overall_accuracy"]
    print(f"\n% --- LiveQA-Bench (Tab:liveqa) ---")
    print(f"    \\textbf{{\\ours}} & \\textbf{{{sa.get('instant', 'TBD')}}} "
          f"& \\textbf{{{sa.get('recent', 'TBD')}}} "
          f"& \\textbf{{{sa.get('historical', 'TBD')}}} "
          f"& \\textbf{{{oa}}} \\\\")

# --- Ablation (from run_docker_eval output) ---
ablation_path = results_path / "ablation_summary.json"
if ablation_path.exists():
    with open(ablation_path) as f:
        abl = json.load(f)
    print(f"\n% --- Ablation (Tab:ablation) ---")
    for key in ["full", "fifo", "no_tqr", "N16", "N32", "N128"]:
        if key in abl:
            print(f"%   {key:10s} -> {abl[key]['overall_accuracy']}%")

# --- Latency ---
lat_path = results_path / "latency.json"
if lat_path.exists():
    with open(lat_path) as f:
        lat = json.load(f)
    print(f"\n% --- Latency (Tab:latency, GPU column) ---")
    for comp, v in lat.items():
        print(f"%   {comp:22s} = {v['mean_ms']:.1f} ms")

# --- Published external numbers (already in the paper) ---
print(f"\n% --- OVO-Bench (Tab:ovobench, published Li et al. CVPR 2025) ---")
print(f"%   Offline:")
print(f"%     Qwen2-VL-7B:        BT=46.5  RT=56.0  FA=48.7  Overall=50.4")
print(f"%     LLaVA-OneVision-7B: BT=43.7  RT=64.0  FA=50.5  Overall=52.7")
print(f"%   Streaming:")
print(f"%     Flash-VStream:      BT=27.4  RT=28.4  FA=45.1  Overall=33.6")
print(f"%     VideoLLM-online:    BT=17.7  RT=20.8  FA=---   Overall=---")
print(f"%     Dispider:           BT=36.1  RT=54.6  FA=34.7  Overall=41.8")
print(f"\n% --- EgoSchema reference (published He et al. CVPR 2025) ---")
print(f"%   Video-LLaVA: 38.4%  LLaVA-Next-Video: 43.9%  Dispider: 55.6%")

print("\n" + "=" * 60)
print("  Copy the LiveQA / ablation / latency values above into")
print("  paper/sec/4_experiments.tex.")
print("  OVO-Bench and EgoSchema numbers are already filled in")
print("  from published sources.")
print("=" * 60)

In [ ]:
# @title Run the dedicated results_to_latex.py script
!cd {EVAL_DIR} && python results_to_latex.py --results-dir {RESULTS_DIR}

## 6. Download Results

Download all result JSONs so you can use them locally.

In [ ]:
# @title List all result files
result_files = sorted(Path(RESULTS_DIR).glob("*.json"))
print(f"Result files ({len(result_files)}):")
for f in result_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:6.1f} KB")

In [ ]:
# @title Download results as zip
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/streammind_results", "zip", RESULTS_DIR)
print(f"Created: {zip_path}")
files.download(zip_path)

In [ ]:
# @title (Optional) Save results to Google Drive
import shutil

DRIVE_SAVE = "/content/drive/MyDrive/streammind_results"
os.makedirs(DRIVE_SAVE, exist_ok=True)

for f in Path(RESULTS_DIR).glob("*.json"):
    shutil.copy2(f, DRIVE_SAVE)
    print(f"  Copied {f.name}")

print(f"\nResults saved to {DRIVE_SAVE}")

## 7. Updating the Paper

After running the evaluation, follow these steps to add results to the paper:

### Step 1: Copy result JSONs to your local machine
Download the zip from Section 6 above and extract into `eval/results/`.

### Step 2: Generate LaTeX values
```bash
cd eval
python results_to_latex.py --results-dir ./results
```

### Step 3: Update `paper/sec/4_experiments.tex`
- **LiveQA-Bench table (Tab:liveqa)**: Replace `\ours` row with Colab-measured values (Phase 2)
- **Ablation table (Tab:ablation)**: Fill in from Phase 3 output
- **Latency table (Tab:latency)**: Replace with actual GPU timings from Phase 1
- **OVO-Bench table (Tab:ovobench)**: Already filled with published numbers from Li et al. (CVPR 2025) — no changes needed
- **EgoSchema reference paragraph**: Already uses published numbers from Dispider paper — no changes needed